# RAG serving on GPU (vLLM) — answer quality + throughput

Runs the **same pipeline as the repo** — only the generator changes: a larger model
served by **vLLM** on a Colab GPU. It reproduces the EM/F1 answer eval with a capable
reader, then benchmarks serving throughput/latency.

**First:** Runtime → Change runtime type → **GPU**.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 1. Install the repo + vLLM


In [ ]:
REPO = "https://github.com/GYOM15/rag-vector-hybrid-graph.git"  # private: use a PAT, or make public
!git clone -q {REPO} rag
%cd rag
!pip install -q -e .
!pip install -q vllm
# spaCy model used by the graph retriever
!pip install -q https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl


## 2. Pick a model that fits the GPU


In [ ]:
import torch
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
# Small model for a T4 (~16 GB); a full 7-8B for L4 / A100.
MODEL = "Qwen/Qwen2.5-3B-Instruct" if gb < 20 else "Qwen/Qwen2.5-7B-Instruct"
print(f"{gb:.0f} GB VRAM -> {MODEL}")


## 3. Serve the model with vLLM (OpenAI-compatible API on :8000)


In [ ]:
import subprocess, time, urllib.request
subprocess.Popen(["vllm", "serve", MODEL, "--port", "8000", "--max-model-len", "8192"])
# Wait until the server answers (weights take a few minutes to load).
for _ in range(180):
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models"); print("vLLM ready"); break
    except Exception:
        time.sleep(5)


## 4. Answer quality (EM/F1) with the served model

Exactly the repo's `eval.answer_eval` — only the backend is now vLLM (set via env).
Compare these numbers to the local 1b/3b baseline in the README.


In [ ]:
import os
os.environ.update({
    "LLM_PROVIDER": "openai",
    "OPENAI_BASE_URL": "http://localhost:8000/v1",
    "OPENAI_MODEL": MODEL,
    "OPENAI_API_KEY": "EMPTY",
})
!python -m eval.answer_eval --max-queries 100


## 5. Serving throughput / latency

With vLLM's continuous batching, throughput should **climb** with concurrency while
latency stays bounded — unlike single-request Ollama (flat throughput, latency grows).


In [ ]:
!python -m eval.serving_bench --base-url http://localhost:8000/v1 --model {MODEL} --n-prompts 64


## 6. (Optional) RAGAS faithfulness, judged by the *served* model — no OpenAI key

RAGAS accepts any OpenAI-compatible endpoint as its judge, so point it at the local
vLLM server (`base_url=http://localhost:8000/v1`, `api_key='EMPTY'`, `model=MODEL`) via a
`langchain_openai.ChatOpenAI`. ragas/langchain versions move fast — if the import fails,
`pip install -U ragas langchain-openai` and retry. Kept as a note (not a fixed cell) to
avoid shipping version-fragile code.


## Results

- `eval/answer_results.json` — EM/F1 with the GPU-served model.
- `eval/serving_results.json` — throughput/latency sweep.

Download them (or push from a clone with write rights) to fold into the repo's README.
